**Procesamiento de Imágenes   ITSE - Instituto Tecnológico de Santiago del Estero**

# Filtering in Image Processing using Masks

# Libraries

In [ ]:
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt

**Explicación:** Se importan las librerías necesarias: `numpy` para operaciones numéricas y manejo de arreglos, `cv2` (OpenCV) para procesamiento de imágenes y filtros, y `matplotlib.pyplot` para visualización de gráficos.

# 1) Basic Examples

In [ ]:
REPO = 'https://raw.githubusercontent.com/domingomery/pr/main/clases/'
!wget {REPO}PR01_Introduccion/datos/rice64.png

**Explicación:** Se descarga la imagen `rice64.png` desde un repositorio en GitHub. Esta imagen se utilizará como ejemplo para los filtros de la sección 1.

## 1.1) Average Mask (Low Pass Filter)

In [ ]:
I = cv.imread('rice64.png',0)
plt.imshow(I, cmap='gray')
plt.title('Imagen Original')
plt.show()
n   = 5
Mk  = np.ones((n,n),np.float32)/n/n
L   = cv.filter2D(I,-1,Mk)
Z = np.concatenate((I,L),axis=1)
plt.figure(figsize=(12,4))
plt.imshow(Z, cmap='gray')
plt.title('Original (izq) | Filtro Promedio (der)')
plt.show()
print('Máscara de promedio 5x5:')
print(Mk)

**Explicación:** Se carga la imagen en escala de grises y se aplica un **filtro de promedio** (filtro pasa bajo) de 5x5. La máscara `Mk` tiene valores iguales (1/25 = 0.04) que promedian los píxeles vecinos. Esto produce un **suavizado** de la imagen, eliminando ruido y detalles finos. Se concatenan original y filtrada para comparar.

## 1.2) Gaussian Mask (Low Pass Filter)

In [ ]:
width  = 15
s      = width/8.5
kernel = cv.getGaussianKernel(width, s)
print('Kernel gaussiano (width x 1):')
print(kernel)
print('Forma del kernel:', kernel.shape)
window = np.outer(kernel, kernel.transpose())
print('Forma de la ventana 2D:', window.shape)
L      = cv.filter2D(I,-1,window)
Z      = np.concatenate((I,L),axis=1)
plt.figure(figsize=(12,4))
plt.imshow(Z, cmap='gray')
plt.title('Original (izq) | Filtro Gaussiano (der)')
plt.show()

**Explicación:** Se aplica un **filtro gaussiano** (filtro pasa bajo) de 15x15 píxeles. A diferencia del filtro de promedio, el gaussiano pondera más los píxeles centrales y menos los distantes, produciendo un suavizado más natural. `cv.getGaussianKernel` genera el kernel 1D y `np.outer` lo convierte en una ventana 2D.

## 1.3) Horizontal Edges

In [ ]:
Id = I.astype(float)

# 3x3 Y-direction kernel
My = np.array([[ 0, -1,  0], [0, 0, 0], [0, 1, 0]])

# Filter the image using filter2D
Gy0 = cv.filter2D(Id, -1, My)

# Para visualización
Gy = np.abs(Gy0)
Gy = Gy/np.max(Gy)*255

Z = np.concatenate((Id,Gy),axis=1)
plt.figure(figsize=(12,4))
plt.imshow(Z, cmap='gray')
plt.title('Original (izq) | Bordes Horizontales (der)')
plt.show()

**Explicación:** Se detectan **bordes horizontales** usando la máscara `My` que resalta los cambios de intensidad en la dirección Y (vertical). Los valores negativos y positivos en la columna central calculan la diferencia entre píxeles superiores e inferiores. Se toman valores absolutos y se normalizan para visualizar mejor.

## 1.4) Vertical Edges

In [ ]:
Id = I.astype(float)

# 3x3 X-direction kernel
Mx = np.array([[ 0, 0, 0], [-1, 0, 1], [ 0, 0, 0]])
print('Máscara de bordes verticales:')
print(Mx)

# Filter the image using filter2D
Gx0 = cv.filter2D(Id, -1, Mx)

Gx = np.abs(Gx0)
Gx = Gx/np.max(Gx)*255

Z = np.concatenate((Id,Gx),axis=1)
plt.figure(figsize=(12,4))
plt.imshow(Z, cmap='gray')
plt.title('Original (izq) | Bordes Verticales (der)')
plt.show()

**Explicación:** Se detectan **bordes verticales** usando la máscara `Mx` que resalta los cambios de intensidad en la dirección X (horizontal). Los valores negativo y positivo en la fila central calculan la diferencia entre píxeles izquierdo y derecho. Se normaliza para visualización.

## 1.5) Both edges (horizontal & vertical)

In [ ]:
G = np.sqrt(np.multiply(Gx0,Gx0) + np.multiply(Gy0,Gy0))
G = G/np.max(G)*255
E = (G>125)*255

Z = np.concatenate((Id,Gx,Gy,G,E),axis=1)
plt.figure(figsize=(18,4))
plt.imshow(Z, cmap='gray')
plt.title('Original | Bordes X | Bordes Y | Magnitud | Binarizado')
plt.show()

**Explicación:** Se combinan los bordes horizontales y verticales calculando la **magnitud del gradiente** `G = sqrt(Gx² + Gy²)`. Luego se **binariza** con umbral 125 para obtener solo los bordes más fuertes. Se muestran 5 imágenes: original, bordes X, bordes Y, magnitud total y bordes binarizados.

## 1.6) Sobel operator edge detection

In [ ]:
Gx0 = cv.Sobel(Id,cv.CV_64F,1,0,ksize=3)
Gy0 = cv.Sobel(Id,cv.CV_64F,0,1,ksize=3)

G = np.sqrt(np.multiply(Gx0,Gx0) + np.multiply(Gy0,Gy0))
G = np.abs(G)
G = G/np.max(G)*255
E = (G>125)*255

Z = np.concatenate((Id,G,E),axis=1)
plt.figure(figsize=(16,4))
plt.imshow(Z, cmap='gray')
plt.title('Original | Magnitud Sobel | Bordes Binarizados')
plt.show()

**Explicación:** El **operador Sobel** es un método de detección de bordes más robusto. Calcula las derivadas parciales en X e Y usando kernels de 3x3 con ponderación gaussiana. Luego se calcula la magnitud del gradiente y se binariza. Es preferido sobre los kernels simples por su mejor respuesta ante ruido.

# 2) High Pass Filter

In [ ]:
REPO = 'https://raw.githubusercontent.com/domingomery/pr/main/clases/'
!wget {REPO}PR01_Introduccion/datos/tokio_color.jpg

**Explicación:** Se descarga la imagen `tokio_color.jpg` que se utilizará para la sección de filtros pasa alto.

In [ ]:
# Input image
Icv = cv.imread('tokio_color.jpg')
plt.figure(figsize=(8,12))
plt.imshow(cv.cvtColor(Icv, cv.COLOR_BGR2RGB))
plt.title('Imagen Original Color')
plt.show()

**Explicación:** Se carga y muestra la imagen original en color. OpenCV carga las imágenes en formato BGR, por lo que se convierte a RGB para mostrarla correctamente con matplotlib.

In [ ]:
# Conversion to grayscale
I = cv.cvtColor(Icv, cv.COLOR_BGR2GRAY)
plt.figure(figsize=(8,12))
plt.imshow(I, cmap='gray')
plt.title('Imagen en Escala de Grises')
plt.show()
print('Dimensiones:', I.shape)

**Explicación:** Se convierte la imagen a **escala de grises** usando `cv.cvtColor` con la conversión `COLOR_BGR2GRAY`. Los filtros de bordes y high-pass se aplican sobre imágenes en escala de grises.

In [ ]:
# Gaussian Filtering (Low Pass)
s      = 39
kernel = cv.getGaussianKernel(s, s/8.5)
window = np.outer(kernel, kernel.transpose())
L      = cv.filter2D(I,-1,window)
Z      = np.concatenate((I,L),axis=1)
plt.figure(figsize=(14,10))
plt.imshow(Z, cmap='gray')
plt.title('Original (izq) | Filtro Gaussiano (der)')
plt.show()

**Explicación:** Se aplica un **filtro gaussiano** de 39x39 píxeles para obtener una versión suavizada `L` de la imagen. Este filtro pasa bajo eliminará los detalles finos y ruido, dejando solo las componentes de baja frecuencia.

In [ ]:
# High Pass Filtering
Id = I.astype(float)
Ld = L.astype(float)
Hd = Id-Ld

# Visualization
H = np.abs(Hd)
H = H-np.min(H)
H = H/np.max(H)*255
Z = np.concatenate((I,H),axis=1)
plt.figure(figsize=(14,10))
plt.imshow(Z, cmap='gray')
plt.title('Original (izq) | Filtro Pasa Alto (der)')
plt.show()

**Explicación:** Se calcula el **filtro pasa alto** restando la imagen suavizada `Ld` de la original `Id`. La imagen resultante `Hd` contiene los detalles o cambios rápidos (alta frecuencia). Se normaliza para visualizar los detalles resaltados.

In [ ]:
# Edge detection using High Pass Filtering
E = (H>50)*255
Z = np.concatenate((I,E),axis=1)
plt.figure(figsize=(14,10))
plt.imshow(Z, cmap='gray')
plt.title('Original (izq) | Bordes con Pasa Alto (der)')
plt.show()

**Explicación:** Se **binariza** la imagen del filtro pasa alto con umbral 50 para obtener una imagen de bordes `E`. Los píxeles con valor mayor a 50 se convierten en blanco (255) y el resto en negro (0).

In [ ]:
# Edge detection using Edge Detection with Sobel
Id = I.astype(float)

Gx0 = cv.Sobel(Id,cv.CV_64F,1,0,ksize=3)
Gy0 = cv.Sobel(Id,cv.CV_64F,0,1,ksize=3)

G = np.sqrt(np.multiply(Gx0,Gx0) + np.multiply(Gy0,Gy0))
G = np.abs(G)
G = G/np.max(G)*255
E = (G>40)*255

Z = np.concatenate((I,G),axis=1)
plt.figure(figsize=(14,10))
plt.imshow(Z, cmap='gray')
plt.title('Original (izq) | Magnitud Sobel (der)')
plt.show()

Z = np.concatenate((I,E),axis=1)
plt.figure(figsize=(14,10))
plt.imshow(Z, cmap='gray')
plt.title('Original (izq) | Bordes Sobel Binarizados (der)')
plt.show()

**Explicación:** Se aplica el **operador Sobel** a la imagen en escala de grises para detectar bordes. Se calculan las derivadas parciales Gx y Gy, luego la magnitud del gradiente G. Se binariza con umbral 40 para obtener los bordes detectados por Sobel.

Another visualization

In [ ]:
L = np.log(G+1)
L = L/np.max(L)*255
Li = L.astype(np.uint8)

Lc = cv.applyColorMap(Li, cv.COLORMAP_PARULA)

plt.figure(figsize=(8,12))
plt.imshow(cv.cvtColor(Lc, cv.COLOR_BGR2RGB))
plt.title('Mapa de Calor - Magnitud Sobel')
plt.show()

**Explicación:** Se aplica una **transformación logarítmica** a la magnitud del gradiente para mejorar el contraste. Luego se convierte a un **mapa de calor** usando `cv.COLORMAP_PARULA` para una visualización colorida de los bordes.

# 3) Sharpen Filtering

This filter is used to enhance the smoothed details

In [ ]:
REPO = 'https://raw.githubusercontent.com/domingomery/pr/main/clases/'
!wget {REPO}PR01_Introduccion/datos/moon.tif

**Explicación:** Se descarga la imagen `moon.tif` que se utilizará para la sección de filtro de realce (sharpen).

In [ ]:
# Original Image
I = cv.imread('moon.tif',0)
plt.figure(figsize=(8,10))
plt.imshow(I, cmap='gray')
plt.title('Imagen Original - Luna')
plt.show()

**Explicación:** Se carga y muestra la imagen original de la luna en escala de grises.

In [ ]:
# Filtered Image (low pass filter)
n   = 5
Mk  = np.ones((n,n),np.float32)/n/n
L   = cv.filter2D(I,-1,Mk)
plt.figure(figsize=(8,10))
plt.imshow(L, cmap='gray')
plt.title('Imagen Filtrada (Filtro Pasa Bajo)')
plt.show()

**Explicación:** Se aplica un **filtro de promedio** (pasa bajo) de 5x5 para obtener una versión suavizada `L` de la imagen. Esta versión filtrada se usará para calcular los detalles.

In [ ]:
# High pass filter
Id = I.astype(float)
Ld = L.astype(float)
Hd = Id-Ld

# Visualization
H = np.abs(Hd)
H = H-np.min(H)
H = H/np.max(H)*255
plt.figure(figsize=(8,10))
plt.imshow(H, cmap='gray')
plt.title('Filtro Pasa Alto (Detalles)')
plt.show()

**Explicación:** Se calcula el **filtro pasa alto** restando la imagen suavizada de la original. La imagen `H` contiene los detalles y texturas de la luna que fueron eliminados por el filtro pasa bajo.

In [ ]:
# Sharpen: high pass pixels are amplified
Jd = Id + 3*Hd

Jd[Jd<0] = 0
Jd = Jd/np.max(Jd)*255

Z = np.concatenate((Id,Jd),axis=1)
print('Original (izq) | Sharpen (der)')
plt.figure(figsize=(14,10))
plt.imshow(Z, cmap='gray')
plt.title('Original (izq) | Imagen Realzada Sharpen (der)')
plt.show()

**Explicación:** Se aplica el **filtro de realce (sharpen)** sumando la imagen original con los detalles amplificados (3 * Hd). Esto acentúa los bordes y texturas, haciendo que la imagen parezca más nítida. Se recortan valores negativos y se normaliza.

---
# EJERCICIO - Actividad 5

## Consignas:
1. Elegir una imagen de la carpeta Clase5
2. Aplicarle suavizado gaussiano y luego el filtro pasa alto para resaltar los detalles
3. Aplicar filtro de sobel para detectar los bordes. Binarizar la imagen

### Paso 1: Descarga de imagen desde Google Drive
Se descarga la imagen `monedas.jpg` desde la carpeta Clase5 del Google Drive.

In [ ]:
# Instalar gdown para descargar archivos desde Google Drive
!pip install gdown -q

In [ ]:
# Descargar monedas.jpg desde Google Drive
import gdown

url = 'https://drive.google.com/drive/folders/11zJfIPGouOPeYPAjtQtGkJLKBtxojCGr'
gdown.download_folder(url, output='.', quiet=False)

In [ ]:
# Cargar y mostrar imagen para el ejercicio
Icv = cv.imread('monedas.jpg')
I = cv.cvtColor(Icv, cv.COLOR_BGR2GRAY)

plt.figure(figsize=(8,12))
plt.imshow(I, cmap='gray')
plt.title('Imagen Original - Ejercicio (monedas.jpg)')
plt.show()
print('Dimensiones:', I.shape)

**Explicación:** Se descarga la imagen `monedas.jpg` desde el Google Drive usando `gdown`, luego se carga con OpenCV y se convierte a escala de grises para poder aplicar los filtros de detección de bordes.

### Paso 2: Suavizado Gaussiano

In [ ]:
# Suavizado gaussiano
s      = 39
kernel = cv.getGaussianKernel(s, s/8.5)
window = np.outer(kernel, kernel.transpose())
L      = cv.filter2D(I,-1,window)

Z = np.concatenate((I,L),axis=1)
plt.figure(figsize=(14,10))
plt.imshow(Z, cmap='gray')
plt.title('Original (izq) | Suavizado Gaussiano (der)')
plt.show()

**Explicación:** Se aplica un **filtro gaussiano** de 39x39 píxeles con sigma = 39/8.5 ≈ 4.59. Este filtro suaviza la imagen eliminando detalles finos y ruido, preparándola para el filtro pasa alto.

### Paso 3: Filtro Pasa Alto (resaltar detalles)

In [ ]:
# Filtro pasa alto
Id = I.astype(float)
Ld = L.astype(float)
Hd = Id - Ld

# Normalización para visualización
H = np.abs(Hd)
H = H - np.min(H)
H = H / np.max(H) * 255

Z = np.concatenate((I,H),axis=1)
plt.figure(figsize=(14,10))
plt.imshow(Z, cmap='gray')
plt.title('Original (izq) | Detalles - Pasa Alto (der)')
plt.show()

**Explicación:** Se calcula el **filtro pasa alto** restando la imagen suavizada `Ld` de la original `Id`. La imagen resultante `H` muestra los detalles y texturas que fueron eliminados por el suavizado gaussiano. Estos son los componentes de alta frecuencia de la imagen.

### Paso 4: Filtro Sobel para detección de bordes

In [ ]:
# Filtro Sobel
Gx0 = cv.Sobel(Id, cv.CV_64F, 1, 0, ksize=3)
Gy0 = cv.Sobel(Id, cv.CV_64F, 0, 1, ksize=3)

# Magnitud del gradiente
G = np.sqrt(np.multiply(Gx0,Gx0) + np.multiply(Gy0,Gy0))
G = np.abs(G)
G = G / np.max(G) * 255

plt.figure(figsize=(14,10))
plt.imshow(G, cmap='gray')
plt.title('Magnitud del Gradiente Sobel')
plt.show()

**Explicación:** Se aplica el **operador Sobel** calculando las derivadas parciales en X (`Gx0`) e Y (`Gy0`). La magnitud del gradiente `G = sqrt(Gx² + Gy²)` indica la intensidad de los bordes en cada píxel.

### Paso 5: Binarización de la imagen de bordes

In [ ]:
# Binarización
E = (G > 40) * 255

Z = np.concatenate((I, G, E), axis=1)
plt.figure(figsize=(18,10))
plt.imshow(Z, cmap='gray')
plt.title('Original | Magnitud Sobel | Bordes Binarizados')
plt.show()

**Explicación:** Se **binariza** la magnitud del gradiente Sobel con umbral 40. Los píxeles con valor mayor a 40 se convierten en blanco (255 = borde) y el resto en negro (0 = no borde). Se muestra la comparación entre original, magnitud Sobel y bordes binarizados.

### Resultado Final del Ejercicio

In [ ]:
# Resumen: Original, Detalles (Pasa Alto), Bordes Sobel Binarizados
H_norm = H / np.max(H) * 255

Z = np.concatenate((I, H_norm, E), axis=1)
plt.figure(figsize=(20,10))
plt.imshow(Z, cmap='gray')
plt.title('Original | Detalles (Pasa Alto) | Bordes Sobel Binarizados')
plt.show()

**Explicación:** Se muestra el resumen final del ejercicio con las tres imágenes: la original, los detalles resaltados con el filtro pasa alto, y los bordes detectados con el filtro Sobel binarizados.

**Bibliografía**  
D. Mery. Fundamentos de Procesamiento de Imágenes Departamento de Ciencia de la Computación. Universidad Católica de Chile, 2024.